# CatVTON — Local File Inference

Run virtual try-on on **local images** (no server, no ngrok, no Cloudinary).  
Just point to your person image, garment image, and optionally a mask.

## How to run on Colab
1. Set runtime to **GPU** (Runtime → Change runtime type → T4 GPU).
2. Run **Cell 1** (clone) and **Cell 2** (install).
3. **RESTART THE RUNTIME** after Cell 2 (Runtime → Restart session). This is
   required so the freshly installed torch/torchvision/detectron2 load cleanly.
4. Run **Cell 3 onward**.

> If `AutoMasker` fails to load (detectron2 build issues), you can still run
> try-on by setting `MASK_IMAGE_PATH` to your own mask in Cell 5.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# Cell 1: Clone repo (skip if already inside it)
# ═══════════════════════════════════════════════════════════════════════════════
import os
if not os.path.exists("model/pipeline.py"):
    !git clone https://github.com/usman9-ai/Virtual-Try-On.git
    os.chdir("Virtual-Try-On")
print(f"Working directory: {os.getcwd()}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# Cell 2: Install dependencies
# ═══════════════════════════════════════════════════════════════════════════════
# IMPORTANT: Do NOT use `pip install -r requirements.txt` — it has an unsolvable
# huggingface_hub/diffusers/peft conflict. We install compatible ranges instead.

# torch + torchvision MUST be a matched pair (mismatched builds cause the
# 'operator torchvision::nms does not exist' error).
!pip install -q torch==2.4.0 torchvision==0.19.0 --index-url https://download.pytorch.org/whl/cu121

# Pin numpy < 2 — numpy 2.x breaks detectron2/opencv compiled extensions.
!pip install -q "numpy<2"

!pip install -q accelerate>=0.31.0 transformers>=4.46.0 diffusers>=0.30.0
!pip install -q huggingface_hub>=0.27.0 peft>=0.14.0 safetensors
!pip install -q opencv-python pillow scipy scikit-image tqdm matplotlib
!pip install -q fvcore av cloudpickle omegaconf pycocotools
!pip install -q xformers>=0.0.26

# DensePose/SCHP auto-masking needs detectron2 built for THIS Python+torch.
# The vendored detectron2/ folder is precompiled for Python 3.9 and will NOT
# load on Colab's Python 3.11+. Rebuild it from source (takes ~3-5 min):
!pip install -q 'git+https://github.com/facebookresearch/detectron2.git'

print("\n\nInstall finished. >>> NOW RESTART THE RUNTIME <<<")
print("Runtime menu -> Restart session, then run from Cell 3 onward.")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# Cell 3: Verify GPU
# ═══════════════════════════════════════════════════════════════════════════════
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB")
else:
    raise RuntimeError("No GPU detected! Enable GPU in your notebook runtime settings.")

torch.cuda.empty_cache()

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# Cell 4: Load CatVTON Pipeline + AutoMasker
# ═══════════════════════════════════════════════════════════════════════════════
import sys
sys.path.insert(0, os.getcwd())

from huggingface_hub import snapshot_download
from diffusers.image_processor import VaeImageProcessor

from model.pipeline import CatVTONPipeline
from model.cloth_masker import AutoMasker
from utils import resize_and_crop, resize_and_padding, init_weight_dtype

# Download CatVTON checkpoint (contains attention adapter + DensePose + SCHP)
print("Downloading CatVTON checkpoint (first run only)...")
repo_ckpt = "zhengchong/CatVTON"
attn_folder = snapshot_download(repo_id=repo_ckpt)
print(f"Checkpoint at: {attn_folder}")

# NOTE: 'runwayml/stable-diffusion-inpainting' was DELETED from HuggingFace in 2024.
# Use a community mirror instead.
BASE_CKPT = "botp/stable-diffusion-v1-5-inpainting"

# Load pipeline
print("Loading CatVTON pipeline...")
pipeline = CatVTONPipeline(
    base_ckpt=BASE_CKPT,
    attn_ckpt=attn_folder,
    attn_ckpt_version="mix",
    weight_dtype=init_weight_dtype("fp16"),
    use_tf32=True,
    device="cuda",
    skip_safety_check=True,
)

# Load AutoMasker (for automatic person/garment mask generation).
# Requires detectron2 built for the current Python+torch (see Cell 2).
# If this fails, you can still run try-on by supplying your own MASK_IMAGE_PATH.
print("Loading AutoMasker (DensePose + SCHP)...")
mask_processor = VaeImageProcessor(
    vae_scale_factor=8,
    do_normalize=False,
    do_binarize=True,
    do_convert_grayscale=True,
)
try:
    automasker = AutoMasker(
        densepose_ckpt=os.path.join(attn_folder, "DensePose"),
        schp_ckpt=os.path.join(attn_folder, "SCHP"),
        device="cuda",
    )
    print("AutoMasker loaded.")
except Exception as e:
    automasker = None
    print(f"AutoMasker FAILED to load: {e}")
    print("You must provide MASK_IMAGE_PATH manually in Cell 5.")

print("All models loaded!")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# Cell 5: Configuration — SET YOUR FILE PATHS HERE
# ═══════════════════════════════════════════════════════════════════════════════

# ┌─────────────────────────────────────────────────────────────────────────────┐
# │ EDIT THESE PATHS to point to your local images                              │
# └─────────────────────────────────────────────────────────────────────────────┘

PERSON_IMAGE_PATH = "path/to/person.jpg"        # Full-body person photo
GARMENT_IMAGE_PATH = "path/to/garment.jpg"      # Garment image (flat-lay or on model)
MASK_IMAGE_PATH = None                           # Set to a .png path, or None for auto-mask

# Garment type: "upper" (tops/shirts), "lower" (pants/skirts), "overall" (dresses/full)
CLOTH_TYPE = "upper"

# Output settings
OUTPUT_DIR = "./results"                         # Where to save results
NUM_INFERENCE_STEPS = 50                         # More steps = better quality (20-50)
GUIDANCE_SCALE = 5.0                             # Higher = more garment fidelity (2.5-7.5)
SEED = 42                                        # For reproducibility (set None for random)

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Config ready. Output will be saved to: {OUTPUT_DIR}/")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# Cell 6: Run Try-On Inference
# ═══════════════════════════════════════════════════════════════════════════════
from PIL import Image
import numpy as np

# --- Load images ---
person = Image.open(PERSON_IMAGE_PATH).convert("RGB")
cloth = Image.open(GARMENT_IMAGE_PATH).convert("RGB")
print(f"Person: {person.size}, Garment: {cloth.size}")

# --- Compute target resolution (from UNet config) ---
W = pipeline.unet.config.sample_size * 8
H = pipeline.unet.config.sample_size * 8
print(f"Processing at: {W}x{H}")

# --- Resize to model resolution ---
person = resize_and_padding(person, (W, H))
cloth = resize_and_padding(cloth, (W, H))

# --- Generate or load mask ---
if MASK_IMAGE_PATH and os.path.exists(MASK_IMAGE_PATH):
    # Use provided mask
    person_mask = Image.open(MASK_IMAGE_PATH).convert("L")
    person_mask = person_mask.resize((W, H), Image.NEAREST)
    print("Using provided mask.")
elif automasker is not None:
    # Auto-generate mask with DensePose + SCHP
    print(f"Auto-generating person mask (cloth_type='{CLOTH_TYPE}')...")
    raw_person_mask = automasker(person, CLOTH_TYPE)["mask"]
    raw_person_mask = resize_and_padding(raw_person_mask, (W, H))
    person_mask = mask_processor.blur(raw_person_mask, blur_factor=4).convert("L")
else:
    raise RuntimeError(
        "No mask available: AutoMasker failed to load AND no MASK_IMAGE_PATH set. "
        "Set MASK_IMAGE_PATH in Cell 5 to a binary mask .png."
    )

print(f"Mask size: {person_mask.size}")

# --- Run inference ---
generator = torch.Generator(device="cuda").manual_seed(SEED) if SEED else None

print(f"Running inference ({NUM_INFERENCE_STEPS} steps, guidance={GUIDANCE_SCALE})...")
images = pipeline(
    image=person,
    condition_image=cloth,
    mask=person_mask,
    num_inference_steps=NUM_INFERENCE_STEPS,
    guidance_scale=GUIDANCE_SCALE,
    generator=generator,
)

result_image = images[0]
print(f"Done! Result size: {result_image.size}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# Cell 7: Display Results
# ═══════════════════════════════════════════════════════════════════════════════
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 4, figsize=(20, 6))

axes[0].imshow(person)
axes[0].set_title("Person")
axes[0].axis("off")

axes[1].imshow(cloth)
axes[1].set_title("Garment")
axes[1].axis("off")

axes[2].imshow(person_mask, cmap="gray")
axes[2].set_title("Mask (auto-generated)")
axes[2].axis("off")

axes[3].imshow(result_image)
axes[3].set_title("Try-On Result")
axes[3].axis("off")

plt.tight_layout()
plt.show()

---
## Real-ESRGAN Enhancement (Cloth Texture & Fold Sharpening)

Enhances the try-on result to recover fine garment detail — fabric texture, seams, folds.  
Use `region_only=True` to sharpen **only the garment** and leave the face/background untouched.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# Cell 8: Real-ESRGAN Enhancement
# ═══════════════════════════════════════════════════════════════════════════════
from model.enhancer import RealESRGANEnhancer

# Load the enhancer once (downloads official weights on first run, ~64 MB)
# scale=4 model gives the sharpest detail; tile bounds VRAM for big images.
enhancer = RealESRGANEnhancer(
    scale=4,          # 4x model (best detail). Use 2 for lighter weight.
    device="cuda",
    half=True,        # fp16 — faster, less VRAM
    tile=512,         # tiled inference to bound VRAM (0 = whole image at once)
    tile_pad=32,
)
print("Real-ESRGAN enhancer loaded.")

# ── Enhancement settings ──────────────────────────────────────────────────────
REGION_ONLY = True    # True = sharpen only the garment region (recommended)
OUTSCALE = 1.0        # 1.0 = same size as result; 2.0 = 2x larger output

if REGION_ONLY:
    # Sharpen only the masked garment area, composite back over the original
    enhanced_image = enhancer.enhance_region(
        result_image, person_mask, outscale=OUTSCALE, feather=8,
    )
    print(f"Region-only enhancement applied. Size: {enhanced_image.size}")
else:
    # Enhance the whole image
    enhanced_image = enhancer.enhance(result_image, outscale=OUTSCALE)
    print(f"Full-image enhancement applied. Size: {enhanced_image.size}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# Cell 9: Compare Before / After Enhancement
# ═══════════════════════════════════════════════════════════════════════════════
fig, axes = plt.subplots(1, 2, figsize=(14, 8))

axes[0].imshow(result_image)
axes[0].set_title("Try-On Result (before)")
axes[0].axis("off")

axes[1].imshow(enhanced_image)
axes[1].set_title("Real-ESRGAN Enhanced (after)")
axes[1].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# Cell 10: Save Results (raw + enhanced)
# ═══════════════════════════════════════════════════════════════════════════════
from pathlib import Path

person_stem = Path(PERSON_IMAGE_PATH).stem
garment_stem = Path(GARMENT_IMAGE_PATH).stem
base_name = f"{person_stem}_x_{garment_stem}"

# Raw try-on result
raw_path = os.path.join(OUTPUT_DIR, f"{base_name}_result.png")
result_image.save(raw_path)
print(f"Raw result saved to:      {raw_path}")

# Enhanced result
enhanced_path = os.path.join(OUTPUT_DIR, f"{base_name}_enhanced.png")
enhanced_image.save(enhanced_path)
print(f"Enhanced result saved to: {enhanced_path}")

# Mask for reference
mask_path = os.path.join(OUTPUT_DIR, f"{person_stem}_mask.png")
person_mask.save(mask_path)
print(f"Mask saved to:            {mask_path}")

---
## Batch Mode: Process Multiple Garments on One Person

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# Cell 11: Batch Inference — Multiple garments on one person (with enhancement)
# ═══════════════════════════════════════════════════════════════════════════════

def run_tryon_local(
    person_path,
    garment_path,
    mask_path=None,
    cloth_type="upper",
    steps=50,
    guidance=5.0,
    seed=42,
    enhance=True,
    region_only=True,
    outscale=1.0,
):
    """
    Run try-on on local files. Returns the (optionally enhanced) result PIL Image.
    
    Args:
        person_path:  Path to person image
        garment_path: Path to garment image
        mask_path:    Path to mask image (L-mode .png), or None for auto-mask
        cloth_type:   "upper", "lower", or "overall"
        steps:        Denoising steps
        guidance:     CFG guidance scale
        seed:         Random seed (None for random)
        enhance:      Apply Real-ESRGAN enhancement (requires Cell 8 enhancer)
        region_only:  Enhance only the garment region
        outscale:     Output scale relative to result
    
    Returns:
        PIL.Image — the try-on result
    """
    person = Image.open(person_path).convert("RGB")
    cloth = Image.open(garment_path).convert("RGB")

    W = pipeline.unet.config.sample_size * 8
    H = pipeline.unet.config.sample_size * 8
    person = resize_and_padding(person, (W, H))
    cloth = resize_and_padding(cloth, (W, H))

    if mask_path and os.path.exists(mask_path):
        pmask = Image.open(mask_path).convert("L").resize((W, H), Image.NEAREST)
    else:
        raw = automasker(person, cloth_type)["mask"]
        raw = resize_and_padding(raw, (W, H))
        pmask = mask_processor.blur(raw, blur_factor=4).convert("L")

    gen = torch.Generator(device="cuda").manual_seed(seed) if seed else None

    images = pipeline(
        image=person,
        condition_image=cloth,
        mask=pmask,
        num_inference_steps=steps,
        guidance_scale=guidance,
        generator=gen,
    )
    out = images[0]

    # Optional Real-ESRGAN enhancement (requires `enhancer` from Cell 8)
    if enhance and "enhancer" in globals():
        if region_only:
            out = enhancer.enhance_region(out, pmask, outscale=outscale, feather=8)
        else:
            out = enhancer.enhance(out, outscale=outscale)
    return out


# ── Example batch usage ───────────────────────────────────────────────────────
# Uncomment and edit the paths below:

# PERSON = "data/person/model_01.jpg"
# GARMENTS = [
#     "data/garment/shirt_blue.jpg",
#     "data/garment/tshirt_white.jpg",
#     "data/garment/jacket_black.jpg",
# ]
#
# for i, gpath in enumerate(GARMENTS):
#     print(f"Processing [{i+1}/{len(GARMENTS)}]: {gpath}")
#     result = run_tryon_local(PERSON, gpath, cloth_type="upper", enhance=True)
#     stem = Path(gpath).stem
#     result.save(f"{OUTPUT_DIR}/batch_{stem}.png")
#     print(f"  Saved: {OUTPUT_DIR}/batch_{stem}.png")
#
# print("Batch complete!")

print("Batch helper ready — uncomment the example above and set your paths.")

---
## Process an Entire Folder

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# Cell 12: Folder-based batch — person/ + garment/ directories
# ═══════════════════════════════════════════════════════════════════════════════

def run_tryon_folder(
    person_dir,
    garment_dir,
    mask_dir=None,
    output_dir="./results",
    cloth_type="upper",
    steps=50,
    guidance=5.0,
    seed=42,
):
    """
    Process all matching person-garment pairs from folders.
    
    Expects same filenames in person_dir/ and garment_dir/.
    Optionally reads masks from mask_dir/ (same filename, .png).
    """
    os.makedirs(output_dir, exist_ok=True)
    
    person_files = sorted([
        f for f in os.listdir(person_dir)
        if f.lower().endswith((".jpg", ".jpeg", ".png"))
    ])
    
    print(f"Found {len(person_files)} person images in {person_dir}")
    
    for i, fname in enumerate(person_files):
        person_path = os.path.join(person_dir, fname)
        garment_path = os.path.join(garment_dir, fname)
        
        if not os.path.exists(garment_path):
            # Try with different extension
            stem = Path(fname).stem
            garment_path = None
            for ext in [".jpg", ".jpeg", ".png"]:
                candidate = os.path.join(garment_dir, stem + ext)
                if os.path.exists(candidate):
                    garment_path = candidate
                    break
            if garment_path is None:
                print(f"  [{i+1}] SKIP (no matching garment): {fname}")
                continue
        
        mask_path = None
        if mask_dir:
            stem = Path(fname).stem
            mp = os.path.join(mask_dir, stem + ".png")
            if os.path.exists(mp):
                mask_path = mp
        
        print(f"  [{i+1}/{len(person_files)}] {fname} ...", end=" ")
        result = run_tryon_local(
            person_path, garment_path, mask_path,
            cloth_type=cloth_type, steps=steps, guidance=guidance, seed=seed,
        )
        
        out_path = os.path.join(output_dir, Path(fname).stem + "_result.png")
        result.save(out_path)
        print(f"saved.")
    
    print(f"\nAll done! Results in: {output_dir}/")


# ── Example usage ─────────────────────────────────────────────────────────────
# Uncomment and edit:

# run_tryon_folder(
#     person_dir="./data/person",
#     garment_dir="./data/garment",
#     mask_dir="./data/mask",       # or None for auto-mask
#     output_dir="./results",
#     cloth_type="upper",
#     steps=50,
#     guidance=5.0,
#     seed=42,
# )

print("Folder processor ready — uncomment above to run on a dataset.")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# Cell 13: VRAM usage info
# ═══════════════════════════════════════════════════════════════════════════════
print(f"VRAM allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
print(f"VRAM reserved:  {torch.cuda.memory_reserved() / 1024**3:.2f} GB")
print(f"VRAM total:     {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB")